In [ ]:
#clean food data set
import pandas as pd
df = pd.read_csv("StateAndCountyData.csv", header=None,
                 names=["FIPS", "State", "County", "Variable", "Value"],
                 dtype={"FIPS": str})
df_ca = df[df["State"] == "CA"]
variables = ["PCT_LACCESS_POP19", "MEDHHINC21", "FFRPTH20"]
df_ca_filtered = df_ca[df_ca["Variable"].isin(variables)]
df_food = df_ca_filtered.pivot(index=["FIPS", "County"],
                                columns="Variable",
                                values="Value").reset_index()
df_food.columns.name = None
df_food[variables] = df_food[variables].astype(float)
df_food = df_food.sort_values("County").reset_index(drop=True)

/tmp/ipykernel_184/3062521547.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("StateAndCountyData.csv", header=None,


In [ ]:
#clean health data set
file = "CountyHealthRankings.xlsx"
df_rank = pd.read_excel(
    file,
    sheet_name="Ranked Measure Data",
    header=1
)
df_add = pd.read_excel(
    file,
    sheet_name="Additional Measure Data",
    header=1
)
df_obesity = df_rank[["FIPS", "County", "% Adults with Obesity"]]
df_diabetes = df_add[["FIPS", "County", "% Adults with Diabetes"]]
df_obesity["FIPS"] = df_obesity["FIPS"].astype(str).str.zfill(5)
df_diabetes["FIPS"] = df_diabetes["FIPS"].astype(str).str.zfill(5)
df_health = pd.merge(
    df_obesity,
    df_diabetes[["FIPS", "% Adults with Diabetes"]],
    on="FIPS",
    how="inner")
df_health

/tmp/ipykernel_184/1834040534.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_obesity["FIPS"] = df_obesity["FIPS"].astype(str).str.zfill(5)
/tmp/ipykernel_184/1834040534.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_diabetes["FIPS"] = df_diabetes["FIPS"].astype(str).str.zfill(5)


,FIPS,County,% Adults with Obesity,% Adults with Diabetes
0,06000,NaN,26.2,9.4
1,06001,Alameda,23.5,9.1
2,06003,Alpine,29.9,9.9
3,06005,Amador,29.4,9.0
4,06007,Butte,31.7,9.5
5,06009,Calaveras,30.1,8.7
6,06011,Colusa,32.3,11.8
7,06013,Contra Costa,23.9,9.4
8,06015,Del Norte,33.4,10.9
9,06017,El Dorado,26.0,7.9


In [ ]:
#merge
import numpy as np
df_final = pd.merge(df_food, df_health, on=["FIPS", "County"])
df_final = df_final.rename(columns={
    "PCT_LACCESS_POP19": "% Population with low access to store",
    "FFRPTH20": "Fast-food restaurants/1,000 population",
    "MEDHHINC21": "Median household income"
})
df_final["Fast-food restaurants/1,000 population"] = (
    df_final["Fast-food restaurants/1,000 population"]
    .replace(-9999.0, np.nan)
)
df_final

,FIPS,County,"Fast-food restaurants/1,000 population",Median household income,% Population with low access to store,% Adults with Obesity,% Adults with Diabetes
0,06001,Alameda,0.827156,108971.0,8.083675,23.5,9.1
1,06003,Alpine,2.680965,87570.0,56.969242,29.9,9.9
2,06005,Amador,0.573809,68159.0,0.174187,29.4,9.0
3,06007,Butte,0.737976,62982.0,19.113434,31.7,9.5
4,06009,Calaveras,0.604647,68298.0,14.736117,30.1,8.7
5,06011,Colusa,0.417478,60725.0,39.329746,32.3,11.8
6,06013,Contra Costa,0.678623,110595.0,23.228312,23.9,9.4
7,06015,Del Norte,0.321796,48108.0,34.407192,33.4,10.9
8,06017,El Dorado,0.673837,87689.0,25.222891,26.0,7.9
9,06019,Fresno,0.731329,63140.0,13.327069,37.6,13.5


In [ ]:
df_final.to_csv('datasci112_final.csv', index=False)
from google.colab import files
files.download('datasci112_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>